# Codebase Workflow Walkthrough

This notebook is a guided tour of the live `src/` code.

It is designed to answer three questions clearly:

1. What is the workflow of the code?
2. What does each layer do?
3. What is already implemented and working?

We start from the simplest direct interaction examples and move upward through the model layer, gauge metadata, gauge compilation, and the convention-fixed covariant and pure-gauge features.

It covers the main workflow of the codebase, but not every regression case that lives in `src/examples.py`.

## Recommended way to read this notebook

Run the notebook from top to bottom.

The order is intentional:

- direct engine with scalars
- add derivatives
- add fermions and tensor structures
- mix sectors
- understand the metadata layer
- build a first `Model`
- compile gauge interactions from metadata
- compile covariant kinetic terms
- compile pure-gauge Yang-Mills structures

That mirrors the actual growth of the codebase.

Quick orientation while reading:

- when the notebook calls `vertex_factor(...)` or simplification helpers, we are using `src/model_symbolica.py`
- when it creates `Field`, `InteractionTerm`, `Model`, or kinetic-term declarations, we are using `src/model.py`
- when it calls `compile_minimal_gauge_interactions(...)` or `compile_covariant_terms(...)`, we are using `src/gauge_compiler.py`

In [1]:
import sys
from pathlib import Path
from fractions import Fraction
from pprint import pprint

root = Path.cwd()
src_path = root / "src"
if not src_path.exists():
    src_path = root.parent / "src"

sys.path.insert(0, str(src_path))

from model_symbolica import (
    S,
    Expression,
    I,
    Delta,
    Dot,
    pcomp,
    vertex_factor,
    simplify_deltas,
    simplify_spinor_indices,
    simplify_vertex,
    infer_derivative_targets,
    compact_vertex_sum_form,
    compact_sum_notation,
)

from spenso_structures import (
    SPINOR_KIND,
    LORENTZ_KIND,
    COLOR_FUND_KIND,
    COLOR_ADJ_KIND,
    gamma_matrix,
    gamma5_matrix,
    gauge_generator,
    structure_constant,
    lorentz_metric,
    simplify_gamma_chain,
)

from operators import (
    psi_bar_gamma_psi,
    psi_bar_gamma5_psi,
    current_current,
    quark_gluon_current,
    scalar_gauge_contact,
)

from model import (
    Field,
    GaugeGroup,
    GaugeRepresentation,
    InteractionTerm,
    DerivativeAction,
    Model,
    DiracKineticTerm,
    ComplexScalarKineticTerm,
    GaugeKineticTerm,
    SPINOR_INDEX,
    LORENTZ_INDEX,
    COLOR_FUND_INDEX,
    COLOR_ADJ_INDEX,
)

from gauge_compiler import (
    compile_minimal_gauge_interactions,
    compile_covariant_terms,
)

print("Python:", sys.executable)
print("src path:", src_path.resolve())

x = S("x")
d = S("d")

def direct_vertex(species_map=None, simplify_gamma=False, strip_externals=True, include_delta=False, **kwargs):
    expr = vertex_factor(
        x=x,
        d=d,
        strip_externals=strip_externals,
        include_delta=include_delta,
        **kwargs,
    )
    expr = simplify_deltas(expr, species_map=species_map)
    q_ = S("q_")
    x_ = S("x_")
    expr = expr.replace(Expression.EXP(-I * Dot(q_, x_)), 1)
    if simplify_gamma:
        expr = simplify_gamma_chain(expr)
    return expr

def model_vertex(interaction, external_legs, species_map=None, strip_externals=True, include_delta=False, simplify_gamma=False):
    expr = vertex_factor(
        interaction=interaction,
        external_legs=external_legs,
        x=x,
        d=d,
        strip_externals=strip_externals,
        include_delta=include_delta,
    )
    expr = simplify_deltas(expr, species_map=species_map)
    q_ = S("q_")
    x_ = S("x_")
    expr = expr.replace(Expression.EXP(-I * Dot(q_, x_)), 1)
    if simplify_gamma:
        expr = simplify_gamma_chain(expr)
    return expr

def show(title, expr):
    print(f"\
=== {title} ===")
    print(expr)

def labels(terms):
    return [term.label for term in terms]

Python: /Users/rems/Documents/MScThesis/.venv/bin/python
src path: /Users/rems/Documents/MScThesis/src


## 0. Workflow in one paragraph

The code has one central engine: `vertex_factor(...)` in `src/model_symbolica.py`.

Around that engine, the current layers are:

- `src/model_symbolica.py`: `vertex_factor(...)`, simplification helpers, derivative bookkeeping helpers
- `src/model.py`: `Field`, `InteractionTerm`, `Model`, and the structured bridge into engine kwargs
- `src/gauge_compiler.py`: `compile_minimal_gauge_interactions(...)` and `compile_covariant_terms(...)`

So everything else either:

- prepares inputs for that engine, or
- builds symbolic tensor structures for the engine, or
- compiles higher-level physics declarations into ordinary `InteractionTerm` objects that the same engine can evaluate.

So the notebook always asks the same question: **what do we feed into `vertex_factor(...)`, and how do we build those inputs more systematically?**

In [2]:
p1, p2, p3, p4 = S("p1", "p2", "p3", "p4")
b1, b2, b3, b4 = S("b1", "b2", "b3", "b4")

phi0, chi0 = S("phi0", "chi0")
phiC0, phiCdag0 = S("phiC0", "phiCdag0")
phiQED0, phiQEDdag0 = S("phiQED0", "phiQEDdag0")
phiQCD0, phiQCDdag0 = S("phiQCD0", "phiQCDdag0")
psi0, psibar0 = S("psi0", "psibar0")
psiQED0, psibarQED0 = S("psiQED0", "psibarQED0")
A0, G0 = S("A0", "G0")

mu, nu, rho, sigma = S("mu", "nu", "rho", "sigma")
mu3, mu4 = S("mu3", "mu4")

lam4, g_sym, gD, yF, gV, eQED, qPsi, qPhi, gS = S(
    "lam4", "g", "gD", "yF", "gV", "eQED", "qPsi", "qPhi", "gS"
)

alpha_s = S("alpha_s")
a = S("a")
i_bar, i_psi = S("i_bar", "i_psi")
i1, i2, i3, i4 = S("i1", "i2", "i3", "i4")
s1, s2, s3, s4 = S("s1", "s2", "s3", "s4")
a3, a4, a5, a6 = S("a3", "a4", "a5", "a6")
c1, c2 = S("c1", "c2")

print("Common symbols initialised.")

Common symbols initialised.


## 1. Direct engine input with pure scalars

The most basic interface is the **direct API**.

We provide:

- `coupling`
- `alphas`: the field species appearing in the interaction monomial
- `betas`: the species attached to the external legs
- `ps`: the external momenta

For bosons this is already enough for the combinatorics.

In [3]:
L_phi4 = {
    "coupling": lam4,
    "alphas": [phi0, phi0, phi0, phi0],
    "betas": [b1, b2, b3, b4],
    "ps": [p1, p2, p3, p4],
}

L_phi2chi2 = {
    "coupling": g_sym,
    "alphas": [phi0, phi0, chi0, chi0],
    "betas": [b1, b2, b3, b4],
    "ps": [p1, p2, p3, p4],
}

print("Direct API dictionary for phi^4:")
pprint(L_phi4)

V_phi4_full = vertex_factor(x=x, d=d, **L_phi4)
V_phi4_reduced = direct_vertex(
    **L_phi4,
    species_map={b1: phi0, b2: phi0, b3: phi0, b4: phi0},
)

V_phi2chi2 = direct_vertex(
    **L_phi2chi2,
    species_map={b1: phi0, b2: phi0, b3: chi0, b4: chi0},
)

show("phi^4 with overall momentum delta", V_phi4_full)
show("phi^4 reduced vertex", V_phi4_reduced)
show("phi^2 chi^2 reduced vertex", V_phi2chi2)

Direct API dictionary for phi^4:
{'alphas': [phi0, phi0, phi0, phi0],
 'betas': [b1, b2, b3, b4],
 'coupling': lam4,
 'ps': [p1, p2, p3, p4]}
=== phi^4 with overall momentum delta ===
24𝑖*lam4*(2*𝜋)^d*delta(b1,phi0)*delta(b2,phi0)*delta(b3,phi0)*delta(b4,phi0)*Delta(p1+p2+p3+p4)
=== phi^4 reduced vertex ===
24𝑖*lam4
=== phi^2 chi^2 reduced vertex ===
4𝑖*g


## 2. Add derivatives

Derivative interactions are encoded by:

- `derivative_indices`: which Lorentz indices appear
- `derivative_targets`: which field slots those derivatives act on

The helper `infer_derivative_targets(...)` builds these lists from a more readable slot-based description.

In [4]:
deriv_indices, deriv_targets = infer_derivative_targets([(0, [mu]), (1, [nu])])

L_deriv = {
    "coupling": gD,
    "alphas": [phi0, phi0, phi0, phi0],
    "betas": [b1, b2, b3, b4],
    "ps": [p1, p2, p3, p4],
    "derivative_indices": deriv_indices,
    "derivative_targets": deriv_targets,
}

V_deriv = direct_vertex(
    **L_deriv,
    species_map={b1: phi0, b2: phi0, b3: phi0, b4: phi0},
)

V_deriv_compact = compact_vertex_sum_form(
    coupling=gD,
    ps=[p1, p2, p3, p4],
    derivative_indices=deriv_indices,
    derivative_targets=deriv_targets,
    d=d,
    field_species=[phi0, phi0, phi0, phi0],
    leg_species=[phi0, phi0, phi0, phi0],
)

print("Derivative indices:", deriv_indices)
print("Derivative targets:", deriv_targets)
print("Compact sigma notation:", compact_sum_notation(derivative_indices=deriv_indices, derivative_targets=deriv_targets, n_legs=4))

show("Expanded derivative vertex", V_deriv)
show("Compact derivative sum form", V_deriv_compact)

Derivative indices: [mu, nu]
Derivative targets: [0, 1]
Compact sigma notation: (2)! * Σ_{a, b distinct} p_{a,mu} p_{b,nu}
=== Expanded derivative vertex ===
1𝑖*(-2*gD*pcomp(p1,mu)*pcomp(p2,nu)-2*gD*pcomp(p1,mu)*pcomp(p3,nu)-2*gD*pcomp(p1,mu)*pcomp(p4,nu)-2*gD*pcomp(p1,nu)*pcomp(p2,mu)-2*gD*pcomp(p1,nu)*pcomp(p3,mu)-2*gD*pcomp(p1,nu)*pcomp(p4,mu)-2*gD*pcomp(p2,mu)*pcomp(p3,nu)-2*gD*pcomp(p2,mu)*pcomp(p4,nu)-2*gD*pcomp(p2,nu)*pcomp(p3,mu)-2*gD*pcomp(p2,nu)*pcomp(p4,mu)-2*gD*pcomp(p3,mu)*pcomp(p4,nu)-2*gD*pcomp(p3,nu)*pcomp(p4,mu))
=== Compact derivative sum form ===
-1𝑖*gD*(2*𝜋)^d*(2*pcomp(p1,mu)*pcomp(p2,nu)+2*pcomp(p1,mu)*pcomp(p3,nu)+2*pcomp(p1,mu)*pcomp(p4,nu)+2*pcomp(p1,nu)*pcomp(p2,mu)+2*pcomp(p1,nu)*pcomp(p3,mu)+2*pcomp(p1,nu)*pcomp(p4,mu)+2*pcomp(p2,mu)*pcomp(p3,nu)+2*pcomp(p2,mu)*pcomp(p4,nu)+2*pcomp(p2,nu)*pcomp(p3,mu)+2*pcomp(p2,nu)*pcomp(p4,mu)+2*pcomp(p3,mu)*pcomp(p4,nu)+2*pcomp(p3,nu)*pcomp(p4,mu))*Delta(p1+p2+p3+p4)


## 3. Add fermions and tensor structures

For fermions we must tell the engine more:

- the statistics are fermionic
- the field roles are `psibar`, `psi`, `scalar`, `vector`, ...
- the spinor labels must be explicit if we want open spinor-index output

This is where the direct engine already starts to look like a real Feynman-rule generator.

In [5]:
L_yukawa = {
    "coupling": yF,
    "alphas": [psibar0, psi0, phi0],
    "betas": [b1, b2, b3],
    "ps": [p1, p2, p3],
    "statistics": "fermion",
    "field_roles": ["psibar", "psi", "scalar"],
    "leg_roles": ["psibar", "psi", "scalar"],
    "field_spinor_indices": [a,a, None],
    "leg_spins": [s1, s2, s3],
}

V_yukawa = direct_vertex(
    **L_yukawa,
    species_map={b1: psibar0, b2: psi0, b3: phi0},
)

L_vec_current = {
    "coupling": gV * gamma_matrix(i_bar, i_psi, mu),
    "alphas": [psibar0, psi0, A0],
    "betas": [b1, b2, b3],
    "ps": [p1, p2, p3],
    "statistics": "fermion",
    "field_roles": ["psibar", "psi", "vector"],
    "leg_roles": ["psibar", "psi", "vector"],
    "field_index_labels": [
        {SPINOR_KIND: i_bar},
        {SPINOR_KIND: i_psi},
        {LORENTZ_KIND: mu},
    ],
    "leg_index_labels": [
        {SPINOR_KIND: i1},
        {SPINOR_KIND: i2},
        {LORENTZ_KIND: mu3},
    ],
    "leg_spins": [s1, s2, s3],
}

V_vec_current = direct_vertex(
    **L_vec_current,
    species_map={b1: psibar0, b2: psi0, b3: A0},
    simplify_gamma=True,
)

show("Yukawa vertex from direct API", V_yukawa)
show("Vector current with explicit open indices", V_vec_current)

=== Yukawa vertex from direct API ===
1𝑖*yF*g(bis(4, i1),bis(4, i2))
=== Vector current with explicit open indices ===
1𝑖*gV*gamma(bis(4, i1),bis(4, i2),mink(4, mu3))


## 4. Mix fermions and derivatives

The engine is not limited to pure scalar derivatives or pure fermion bilinears.

It can also handle mixed operators such as

$$ y_F \, \bar\psi\psi\,(\partial_\mu \phi)(\partial_\nu \chi). $$

In [6]:
L_mix = {
    "coupling": yF,
    "alphas": [psibar0, psi0, phi0, chi0],
    "betas": [b1, b2, b3, b4],
    "ps": [p1, p2, p3, p4],
    "derivative_indices": [mu, nu],
    "derivative_targets": [2, 3],
    "statistics": "fermion",
    "field_roles": ["psibar", "psi", "scalar", "scalar"],
    "leg_roles": ["psibar", "psi", "scalar", "scalar"],
    "field_spinor_indices": [alpha_s, alpha_s, None, None],
    "leg_spins": [s1, s2, s3, s4],
}

V_mix = direct_vertex(
    **L_mix,
    species_map={b1: psibar0, b2: psi0, b3: phi0, b4: chi0},
)

show("Mixed fermion-scalar derivative operator", V_mix)

=== Mixed fermion-scalar derivative operator ===
-1𝑖*yF*g(bis(4, i1),bis(4, i2))*pcomp(p3,mu)*pcomp(p4,nu)


## 5. First metadata objects: `Field`, `FieldOccurrence`, `ExternalLeg`, `InteractionTerm`

The direct API is useful, but the codebase really wants to move toward structured model input.

The core metadata objects are:

- `Field`: one declared particle field
- `FieldOccurrence`: one appearance of that field inside an interaction term
- `ExternalLeg`: one external state used when evaluating a vertex
- `InteractionTerm`: one monomial in the Lagrangian

The key bridge method is `InteractionTerm.to_vertex_kwargs(...)`: it translates the structured model layer into the parallel-list format expected by the engine.

In [7]:
PhiField = Field("Phi", spin=0, self_conjugate=True, symbol=phi0)
ChiField = Field("Chi", spin=0, self_conjugate=True, symbol=chi0)
PsiField = Field(
    "Psi",
    spin=Fraction(1, 2),
    self_conjugate=False,
    symbol=psi0,
    conjugate_symbol=psibar0,
    indices=(SPINOR_INDEX,),
)
GaugeField = Field("A", spin=1, self_conjugate=True, symbol=A0, indices=(LORENTZ_INDEX,))

phi_occ = PhiField.occurrence()
psibar_occ = PsiField.occurrence(conjugated=True, labels={SPINOR_KIND: alpha_s})
psi_leg = PsiField.leg(p2, spin=s2, labels={SPINOR_KIND: i2})

print("Example FieldOccurrence:", psibar_occ)
print("  role   =", psibar_occ.role)
print("  species=", psibar_occ.species)
print()
print("Example ExternalLeg:", psi_leg)
print("  role            =", psi_leg.role)
print("  effective species=", psi_leg.effective_species)

TERM_phi4 = InteractionTerm(
    coupling=lam4,
    fields=tuple(PhiField.occurrence() for _ in range(4)),
    label="lam4 * phi^4",
)
LEGS_phi4 = tuple(PhiField.leg(p, species=b) for p, b in [(p1, b1), (p2, b2), (p3, b3), (p4, b4)])

TERM_yukawa = InteractionTerm(
    coupling=yF,
    fields=(
        PsiField.occurrence(conjugated=True, labels={SPINOR_KIND: alpha_s}),
        PsiField.occurrence(labels={SPINOR_KIND: alpha_s}),
        PhiField.occurrence(),
    ),
    label="yF * psibar psi phi",
)

LEGS_yukawa = (
    PsiField.leg(p1, conjugated=True, spin=s1, labels={SPINOR_KIND: i1}),
    PsiField.leg(p2, spin=s2, labels={SPINOR_KIND: i2}),
    PhiField.leg(p3, species=b3),
)

print("Keys produced by InteractionTerm.to_vertex_kwargs(...):")
pprint(sorted(TERM_yukawa.to_vertex_kwargs(LEGS_yukawa).keys()))

V_phi4_model = model_vertex(
    TERM_phi4,
    LEGS_phi4,
    species_map={b1: phi0, b2: phi0, b3: phi0, b4: phi0},
)

V_yukawa_model = model_vertex(
    TERM_yukawa,
    LEGS_yukawa,
    species_map={b3: phi0},
)

show("phi^4 through the model layer", V_phi4_model)
show("Yukawa through the model layer", V_yukawa_model)

assert V_yukawa_model.expand().to_canonical_string() == V_yukawa.expand().to_canonical_string()
print("\
Direct Yukawa and model-layer Yukawa agree.")

Example FieldOccurrence: FieldOccurrence(field=Field(name='Psi', spin=Fraction(1, 2), self_conjugate=False, indices=(IndexType(name='Spinor', representation=Representation { rep: SelfDual(1), dim: Concrete(4) }, kind='spinor', prefix='i'),), kind='fermion', statistics='fermion', symbol=psi0, conjugate_symbol=psibar0, mass=None, quantum_numbers={}), conjugated=True, labels={'spinor': alpha_s})
  role   = FieldRole('psibar')
  species= psibar0

Example ExternalLeg: ExternalLeg(field=Field(name='Psi', spin=Fraction(1, 2), self_conjugate=False, indices=(IndexType(name='Spinor', representation=Representation { rep: SelfDual(1), dim: Concrete(4) }, kind='spinor', prefix='i'),), kind='fermion', statistics='fermion', symbol=psi0, conjugate_symbol=psibar0, mass=None, quantum_numbers={}), momentum=p2, conjugated=False, species=None, spin=s2, labels={'spinor': i2})
  role            = FieldRole('psi')
  effective species= psi0
Keys produced by InteractionTerm.to_vertex_kwargs(...):
['alphas',
 'b

## 6. First `Model` object

A `Model` is the top-level container for a theory.

At minimum it can already store:

- fields
- explicit interactions
- gauge groups
- later: covariant terms and gauge kinetic terms

In [8]:
ToyScalarModel = Model(
    name="Toy scalar model",
    fields=(PhiField, ChiField),
    interactions=(TERM_phi4,),
)

print(ToyScalarModel)
print()
print("find_field('Phi') ->", ToyScalarModel.find_field("Phi"))
print("first stored interaction label ->", ToyScalarModel.interactions[0].label)

Model(name='Toy scalar model', gauge_groups=(), fields=(Field(name='Phi', spin=0, self_conjugate=True, indices=(), kind='scalar', statistics='boson', symbol=phi0, conjugate_symbol=None, mass=None, quantum_numbers={}), Field(name='Chi', spin=0, self_conjugate=True, indices=(), kind='scalar', statistics='boson', symbol=chi0, conjugate_symbol=None, mass=None, quantum_numbers={})), parameters=(), interactions=(InteractionTerm(coupling=lam4, fields=(FieldOccurrence(field=Field(name='Phi', spin=0, self_conjugate=True, indices=(), kind='scalar', statistics='boson', symbol=phi0, conjugate_symbol=None, mass=None, quantum_numbers={}), conjugated=False, labels={}), FieldOccurrence(field=Field(name='Phi', spin=0, self_conjugate=True, indices=(), kind='scalar', statistics='boson', symbol=phi0, conjugate_symbol=None, mass=None, quantum_numbers={}), conjugated=False, labels={}), FieldOccurrence(field=Field(name='Phi', spin=0, self_conjugate=True, indices=(), kind='scalar', statistics='boson', symbol=

## 7. Gauge metadata: `GaugeRepresentation` and `GaugeGroup`

To move beyond handwritten interaction terms, the code introduces gauge metadata.

- `GaugeRepresentation` tells the compiler how to build the generator tensor for one matter representation
- `GaugeGroup` defines the symmetry itself: coupling, gauge boson, structure constants, charge label, and supported matter representations

This is the basis for the gauge compiler.

In [9]:
PhiQEDField = Field(
    "PhiQED",
    spin=0,
    self_conjugate=False,
    symbol=phiQED0,
    conjugate_symbol=phiQEDdag0,
    quantum_numbers={"Q": qPhi},
)

PsiQEDField = Field(
    "PsiQED",
    spin=Fraction(1, 2),
    self_conjugate=False,
    symbol=psiQED0,
    conjugate_symbol=psibarQED0,
    indices=(SPINOR_INDEX,),
    quantum_numbers={"Q": qPsi},
)

PhiQCDField = Field(
    "PhiQCD",
    spin=0,
    self_conjugate=False,
    symbol=phiQCD0,
    conjugate_symbol=phiQCDdag0,
    indices=(COLOR_FUND_INDEX,),
)

QuarkField = Field(
    "q",
    spin=Fraction(1, 2),
    self_conjugate=False,
    symbol=psi0,
    conjugate_symbol=psibar0,
    indices=(SPINOR_INDEX, COLOR_FUND_INDEX),
)

GluonField = Field(
    "G",
    spin=1,
    self_conjugate=True,
    symbol=G0,
    indices=(LORENTZ_INDEX, COLOR_ADJ_INDEX),
)

COLOR_FUND_REP = GaugeRepresentation(
    index=COLOR_FUND_INDEX,
    generator_builder=gauge_generator,
    name="fundamental",
)

QED_GROUP = GaugeGroup(
    name="U1QED",
    abelian=True,
    coupling=eQED,
    gauge_boson="A",
    charge="Q",
)

QCD_GROUP = GaugeGroup(
    name="SU3C",
    abelian=False,
    coupling=gS,
    gauge_boson=G0,
    structure_constant=structure_constant,
    representations=(COLOR_FUND_REP,),
)

MODEL_QED_BASE = Model(
    name="FermionQED-minimal",
    gauge_groups=(QED_GROUP,),
    fields=(PsiQEDField, GaugeField),
)

MODEL_QCD_BASE = Model(
    name="QCD-minimal",
    gauge_groups=(QCD_GROUP,),
    fields=(QuarkField, GluonField),
)

print("QCD representation seen by the quark ->", QCD_GROUP.matter_representation(QuarkField))
print("Gauge boson resolved for QED ->", MODEL_QED_BASE.gauge_boson_field(QED_GROUP))

QCD representation seen by the quark -> GaugeRepresentation(index=IndexType(name='ColorFund', representation=Representation { rep: Dualizable(3), dim: Concrete(3) }, kind='color_fund', prefix='c'), generator_builder=<function gauge_generator at 0x10c92dc70>, name='fundamental', slot=None, slot_policy='unique')
Gauge boson resolved for QED -> Field(name='A', spin=1, self_conjugate=True, indices=(IndexType(name='Lorentz', representation=Representation { rep: InlineMetric(0), dim: Concrete(4) }, kind='lorentz', prefix='mu'),), kind='vector', statistics='boson', symbol=A0, conjugate_symbol=None, mass=None, quantum_numbers={})


## 8. Minimal gauge compiler

The **minimal gauge compiler** turns gauge metadata into ordinary `InteractionTerm` objects.

It is a structural compiler: it inserts the right charges, generators, and spectator identities, but it is not yet the convention-fixed covariant expansion of full kinetic terms.

This section uses:

- `GaugeRepresentation`, `GaugeGroup`, `Model`, and `InteractionTerm` from `src/model.py`
- `compile_minimal_gauge_interactions(...)` from `src/gauge_compiler.py`

In [10]:
compiled_qed = compile_minimal_gauge_interactions(MODEL_QED_BASE)
compiled_qcd = compile_minimal_gauge_interactions(MODEL_QCD_BASE)

print("Minimal QED terms:", labels(compiled_qed))
print("Minimal QCD terms:", labels(compiled_qcd))

LEGS_qed = (
    PsiQEDField.leg(p1, conjugated=True, spin=s1, labels={SPINOR_KIND: i1}),
    PsiQEDField.leg(p2, spin=s2, labels={SPINOR_KIND: i2}),
    GaugeField.leg(p3, labels={LORENTZ_KIND: mu3}),
)

LEGS_quark_gluon = (
    QuarkField.leg(p1, conjugated=True, spin=s1, labels={SPINOR_KIND: i1, COLOR_FUND_KIND: c1}),
    QuarkField.leg(p2, spin=s2, labels={SPINOR_KIND: i2, COLOR_FUND_KIND: c2}),
    GluonField.leg(p3, labels={LORENTZ_KIND: mu3, COLOR_ADJ_KIND: a3}),
)

V_qed_min = model_vertex(compiled_qed[0], LEGS_qed, simplify_gamma=True)
V_qcd_min = model_vertex(compiled_qcd[0], LEGS_quark_gluon, simplify_gamma=True)

show("Minimal gauge compiler: fermion QED current", V_qed_min)
show("Minimal gauge compiler: quark-gluon current", V_qcd_min)

Minimal QED terms: ['U1QED: PsiQED gauge current']
Minimal QCD terms: ['SU3C: q gauge current']
=== Minimal gauge compiler: fermion QED current ===
1𝑖*eQED*qPsi*gamma(bis(4, i1),bis(4, i2),mink(4, mu3))
=== Minimal gauge compiler: quark-gluon current ===
1𝑖*gS*gamma(bis(4, i1),bis(4, i2),mink(4, mu3))*t(coad(8, a3),cof(3, c1),cof(3, c2))


## 8.5 Repeated same-kind index slots

The current code can now handle fields that carry the same index kind more than once, as long as the active gauge slot is declared explicitly.

This uses:

- `Field(...)` and `GaugeRepresentation(slot=...)` from `src/model.py`
- `compile_minimal_gauge_interactions(...)` from `src/gauge_compiler.py`

If the representation is ambiguous, the compiler now rejects the model instead of silently collapsing repeated slots.

In [11]:
phiBi0, phiBidag0 = S("phiBi0", "phiBidag0")
c3, c4 = S("c3", "c4")

PhiBiField = Field(
    "PhiBi",
    spin=0,
    self_conjugate=False,
    symbol=phiBi0,
    conjugate_symbol=phiBidag0,
    indices=(COLOR_FUND_INDEX, COLOR_FUND_INDEX),
)

COLOR_FUND_REP_SLOT0 = GaugeRepresentation(
    index=COLOR_FUND_INDEX,
    generator_builder=gauge_generator,
    name="fundamental_slot0",
    slot=0,
)

QCD_GROUP_BISLOT = GaugeGroup(
    name="SU3CBi",
    abelian=False,
    coupling=gS,
    gauge_boson=G0,
    structure_constant=structure_constant,
    representations=(COLOR_FUND_REP_SLOT0,),
)

QCD_GROUP_AMBIGUOUS = GaugeGroup(
    name="SU3CAmbiguous",
    abelian=False,
    coupling=gS,
    gauge_boson=G0,
    structure_constant=structure_constant,
    representations=(COLOR_FUND_REP,),
)

MODEL_BISLOT_OK = Model(
    name="ScalarQCD-bislot-minimal",
    gauge_groups=(QCD_GROUP_BISLOT,),
    fields=(PhiBiField, GluonField),
)

MODEL_BISLOT_BAD = Model(
    name="ScalarQCD-bislot-ambiguous",
    gauge_groups=(QCD_GROUP_AMBIGUOUS,),
    fields=(PhiBiField, GluonField),
)

try:
    compile_minimal_gauge_interactions(MODEL_BISLOT_BAD)
except ValueError as exc:
    print("Ambiguous repeated-slot model rejected:")
    print(" ", exc)

compiled_bislot = compile_minimal_gauge_interactions(MODEL_BISLOT_OK)
print("Repeated-slot compiled labels:", labels(compiled_bislot))

LEGS_bislot = (
    PhiBiField.leg(p1, conjugated=True, species=b1, labels={COLOR_FUND_KIND: (c1, c3)}),
    PhiBiField.leg(p2, species=b2, labels={COLOR_FUND_KIND: (c2, c4)}),
    GluonField.leg(p3, labels={LORENTZ_KIND: mu3, COLOR_ADJ_KIND: a3}, species=b3),
)

V_bislot = (
    model_vertex(compiled_bislot[0], LEGS_bislot, species_map={b1: phiBidag0, b2: phiBi0, b3: G0})
    + model_vertex(compiled_bislot[1], LEGS_bislot, species_map={b1: phiBidag0, b2: phiBi0, b3: G0})
)

show("Repeated-slot scalar gauge current (slot 1 active, slot 2 spectator)", V_bislot)

Ambiguous repeated-slot model rejected:
  Field 'PhiBi' carries repeated index type 'ColorFund'; declare GaugeRepresentation(slot=...) for strict selection, or set GaugeRepresentation(slot_policy='sum') to sum over all matching slots.
Repeated-slot compiled labels: ['SU3CBi: scalar current (+)_slot1', 'SU3CBi: scalar current (-)_slot1', 'SU3CBi: scalar contact [slots 1,1]']
=== Repeated-slot scalar gauge current (slot 1 active, slot 2 spectator) ===
-gS*g(cof(3, c3),cof(3, c4))*t(coad(8, a3),cof(3, c1),cof(3, c2))*pcomp(p1,mu)+gS*g(cof(3, c3),cof(3, c4))*t(coad(8, a3),cof(3, c1),cof(3, c2))*pcomp(p2,mu)


## 9. Convention-fixed kinetic terms and the covariant compiler

The newer physics-facing layer is the **convention-fixed compiler**.

Instead of hand-declaring each interaction, we declare standard kinetic structures such as:

- `DiracKineticTerm(field=...)` for $\bar\psi i \gamma^\mu D_\mu \psi$
- `ComplexScalarKineticTerm(field=...)` for $(D_\mu \phi)^\dagger (D^\mu \phi)$

The compiler then expands these according to the fixed conventions of the project.

This section uses the declaration classes from `src/model.py` and `compile_covariant_terms(...)` from `src/gauge_compiler.py`.

In [12]:
MODEL_QED_FERMION_COV = Model(
    name="FermionQED-covariant",
    gauge_groups=(QED_GROUP,),
    fields=(PsiQEDField, GaugeField),
    covariant_terms=(DiracKineticTerm(field=PsiQEDField),),
)

compiled_qed_cov = compile_covariant_terms(MODEL_QED_FERMION_COV)
print("Covariant fermion-QED terms:", labels(compiled_qed_cov))

V_qed_cov = model_vertex(compiled_qed_cov[0], LEGS_qed, simplify_gamma=True)

print("=====Dirac kinetic term (QED)=====")
show("Vertex q̄ q γ from i q̄ γ^μ D_μ q  [expected: -i e Q γ^μ]", V_qed_cov)

MODEL_SCALAR_QED_COV = Model(
    name="ScalarQED-covariant",
    gauge_groups=(QED_GROUP,),
    fields=(PhiQEDField, GaugeField),
    covariant_terms=(ComplexScalarKineticTerm(field=PhiQEDField),),
)

compiled_sqed = compile_covariant_terms(MODEL_SCALAR_QED_COV)
print("Covariant scalar-QED terms:", labels(compiled_sqed))

LEGS_sqed_current = (
    PhiQEDField.leg(p1, conjugated=True, species=b1),
    PhiQEDField.leg(p2, species=b2),
    GaugeField.leg(p3, labels={LORENTZ_KIND: mu3}, species=b3),
)

LEGS_sqed_contact = (
    PhiQEDField.leg(p1, conjugated=True, species=b1),
    PhiQEDField.leg(p2, species=b2),
    GaugeField.leg(p3, labels={LORENTZ_KIND: mu3}, species=b3),
    GaugeField.leg(p4, labels={LORENTZ_KIND: mu4}, species=b4),
)

sqed_species_3 = {b1: phiQEDdag0, b2: phiQED0, b3: A0}
sqed_species_4 = {b1: phiQEDdag0, b2: phiQED0, b3: A0, b4: A0}

V_sqed_current = (
    model_vertex(compiled_sqed[0], LEGS_sqed_current, species_map=sqed_species_3)
    + model_vertex(compiled_sqed[1], LEGS_sqed_current, species_map=sqed_species_3)
)

V_sqed_contact = model_vertex(compiled_sqed[2], LEGS_sqed_contact, species_map=sqed_species_4)

print("=====Complex-scalar kinetic term (QED) — current=====")
show("Vertex φ† φ A from (D_μ φ)†(D^μ φ) current  [expected: i e Q (p_2-p_1)^μ]", V_sqed_current)
print("=====Complex-scalar kinetic term (QED) — contact=====")
show("Vertex φ† φ A A from (D_μ φ)†(D^μ φ) contact  [expected: 2 i e^2 Q^2 g^{μν}]", V_sqed_contact)

Covariant fermion-QED terms: ['i PsiQEDbar gamma^mu D_mu PsiQED']
=====Dirac kinetic term (QED)=====
=== Vertex q̄ q γ from i q̄ γ^μ D_μ q  [expected: -i e Q γ^μ] ===
-1𝑖*eQED*qPsi*gamma(bis(4, i1),bis(4, i2),mink(4, mu3))
Covariant scalar-QED terms: ['(D_mu PhiQED)^dagger (D^mu PhiQED) U1QED: scalar current (+)', '(D_mu PhiQED)^dagger (D^mu PhiQED) U1QED: scalar current (-)', '(D_mu PhiQED)^dagger (D^mu PhiQED) U1QED: scalar contact']
=====Complex-scalar kinetic term (QED) — current=====
=== Vertex φ† φ A from (D_μ φ)†(D^μ φ) current  [expected: i e Q (p_2-p_1)^μ] ===
-1𝑖*eQED*qPhi*pcomp(p1,mu)+1𝑖*eQED*qPhi*pcomp(p2,mu)
=====Complex-scalar kinetic term (QED) — contact=====
=== Vertex φ† φ A A from (D_μ φ)†(D^μ φ) contact  [expected: 2 i e^2 Q^2 g^{μν}] ===
2𝑖*eQED^2*qPhi^2*g(mink(4, mu3),mink(4, mu4))


## 9.1 Covariant derivative compiler (matter kinetic terms)

In this codebase, “covariant derivative” means the **gauge-theory covariant derivative**.

### Conventions (frozen for the physical compiler)

- **Fourier-space derivative rule**: \(\partial_\mu \to -i\,p_\mu\)
- **Overall vertex factor**: `vertex_factor(...)` contributes the universal overall **`+i`**.
- **Matter covariant derivative**:

\[
D_\mu = \partial_\mu + i g A_\mu
\]

### What `compile_covariant_terms(...)` actually does

The covariant compiler in `src/gauge_compiler.py` takes high-level kinetic-term declarations:

- `DiracKineticTerm(field=...)` for \(\bar\psi i\gamma^\mu D_\mu\psi\)
- `ComplexScalarKineticTerm(field=...)` for \((D_\mu\phi)^\dagger(D^\mu\phi)\)

and expands **only the gauge-interaction part** into ordinary `InteractionTerm` objects.

Those `InteractionTerm`s are then evaluated by the same `vertex_factor(...)` engine as any other interaction.

### Repeated same-kind index slots and `slot_policy`

If a field carries the same index type more than once (e.g. two identical color-fundamental slots), gauge generators can be ambiguous.

Rule:

- **ambiguity → error** by default (`slot_policy="unique"`)
- **explicit opt-in** for full tensor-product semantics (`slot_policy="sum"`)

In [13]:
# --- Fermion QCD: qbar i gamma^mu D_mu q ---
MODEL_QCD_FERMION_COV = Model(
    name="QCD-covariant",
    gauge_groups=(QCD_GROUP,),
    fields=(QuarkField, GluonField),
    covariant_terms=(DiracKineticTerm(field=QuarkField),),
)
compiled_qcd_cov = compile_covariant_terms(MODEL_QCD_FERMION_COV)
print("Covariant fermion-QCD terms:")
for lab in labels(compiled_qcd_cov):
    print(" -", lab)

V_qcd_cov = model_vertex(compiled_qcd_cov[0], LEGS_quark_gluon, simplify_gamma=True)
print("=====Dirac kinetic term (QCD)=====")
show("Vertex q̄ q G from i q̄ γ^μ D_μ q  [expected: -i g_S γ^μ T^a]", V_qcd_cov)


Covariant fermion-QCD terms:
 - i qbar gamma^mu D_mu q [ColorFund]
=====Dirac kinetic term (QCD)=====
=== Vertex q̄ q G from i q̄ γ^μ D_μ q  [expected: -i g_S γ^μ T^a] ===
-1𝑖*gS*gamma(bis(4, i1),bis(4, i2),mink(4, mu3))*t(coad(8, a3),cof(3, c1),cof(3, c2))


In [14]:
# --- Scalar QCD: (D_mu PhiQCD)^dagger (D^mu PhiQCD) ---
MODEL_SCALAR_QCD_COV = Model(
    name="ScalarQCD-covariant",
    gauge_groups=(QCD_GROUP,),
    fields=(PhiQCDField, GluonField),
    covariant_terms=(ComplexScalarKineticTerm(field=PhiQCDField),),
)
compiled_sqcd = compile_covariant_terms(MODEL_SCALAR_QCD_COV)
print("\nCovariant scalar-QCD terms:")
for lab in labels(compiled_sqcd):
    print(" -", lab)

LEGS_sqcd_current = (
    PhiQCDField.leg(p1, conjugated=True, species=b1, labels={COLOR_FUND_KIND: c1}),
    PhiQCDField.leg(p2, species=b2, labels={COLOR_FUND_KIND: c2}),
    GluonField.leg(p3, labels={LORENTZ_KIND: mu3, COLOR_ADJ_KIND: a3}, species=b3),
)
LEGS_sqcd_contact = (
    PhiQCDField.leg(p1, conjugated=True, species=b1, labels={COLOR_FUND_KIND: c1}),
    PhiQCDField.leg(p2, species=b2, labels={COLOR_FUND_KIND: c2}),
    GluonField.leg(p3, labels={LORENTZ_KIND: mu3, COLOR_ADJ_KIND: a3}, species=b3),
    GluonField.leg(p4, labels={LORENTZ_KIND: mu4, COLOR_ADJ_KIND: a4}, species=b4),
)

sqcd_species_3 = {b1: phiQCDdag0, b2: phiQCD0, b3: G0}
sqcd_species_4 = {b1: phiQCDdag0, b2: phiQCD0, b3: G0, b4: G0}

V_sqcd_current = (
    model_vertex(compiled_sqcd[0], LEGS_sqcd_current, species_map=sqcd_species_3)
    + model_vertex(compiled_sqcd[1], LEGS_sqcd_current, species_map=sqcd_species_3)
).expand()
V_sqcd_contact = model_vertex(compiled_sqcd[2], LEGS_sqcd_contact, species_map=sqcd_species_4)
print("=====Complex-scalar kinetic term (QCD) — current=====")
show("Vertex φ† φ G from (D_μ φ)†(D^μ φ) current  [expected: i g_S (p_2-p_1)^μ T^a]", V_sqcd_current)
print("=====Complex-scalar kinetic term (QCD) — contact=====")
show("Vertex φ† φ G G from (D_μ φ)†(D^μ φ) contact  [expected: i g_S^2 g^{μν} × (T^a T^b + T^b T^a) structure]", V_sqcd_contact)


Covariant scalar-QCD terms:
 - (D_mu PhiQCD)^dagger (D^mu PhiQCD) SU3C: scalar current (+)
 - (D_mu PhiQCD)^dagger (D^mu PhiQCD) SU3C: scalar current (-)
 - (D_mu PhiQCD)^dagger (D^mu PhiQCD) SU3C: scalar contact [slots 1,1]
=====Complex-scalar kinetic term (QCD) — current=====
=== Vertex φ† φ G from (D_μ φ)†(D^μ φ) current  [expected: i g_S (p_2-p_1)^μ T^a] ===
-1𝑖*gS*t(coad(8, a3),cof(3, c1),cof(3, c2))*pcomp(p1,mu)+1𝑖*gS*t(coad(8, a3),cof(3, c1),cof(3, c2))*pcomp(p2,mu)
=====Complex-scalar kinetic term (QCD) — contact=====
=== Vertex φ† φ G G from (D_μ φ)†(D^μ φ) contact  [expected: i g_S^2 g^{μν} × (T^a T^b + T^b T^a) structure] ===
1𝑖*(gS^2*g(mink(4, mu3),mink(4, mu4))*t(coad(8, a3),cof(3, c1),cof(3, c_mid_PhiQCD_SU3C))*t(coad(8, a4),cof(3, c_mid_PhiQCD_SU3C),cof(3, c2))+gS^2*g(mink(4, mu3),mink(4, mu4))*t(coad(8, a3),cof(3, c_mid_PhiQCD_SU3C),cof(3, c2))*t(coad(8, a4),cof(3, c1),cof(3, c_mid_PhiQCD_SU3C)))


In [15]:
# --- Mixed QCD + QED fermion kinetic term expands into separate gauge pieces ---
qMix = S("qMix")
PsiMixField = Field(
    "PsiMix",
    spin=Fraction(1, 2),
    self_conjugate=False,
    symbol=psi0,
    conjugate_symbol=psibar0,
    indices=(SPINOR_INDEX, COLOR_FUND_INDEX),
    quantum_numbers={"Q": qMix},
)
MODEL_MIXED_COV = Model(
    name="MixedQCDQED-covariant",
    gauge_groups=(QCD_GROUP, QED_GROUP),
    fields=(PsiMixField, GluonField, GaugeField),
    covariant_terms=(DiracKineticTerm(field=PsiMixField),),
)
compiled_mixed = compile_covariant_terms(MODEL_MIXED_COV)
print("\nMixed-group Dirac kinetic term expands into:")
for lab in labels(compiled_mixed):
    print(" -", lab)

LEGS_mixed_gluon = (
    PsiMixField.leg(p1, conjugated=True, spin=s1, labels={SPINOR_KIND: i1, COLOR_FUND_KIND: c1}),
    PsiMixField.leg(p2, spin=s2, labels={SPINOR_KIND: i2, COLOR_FUND_KIND: c2}),
    GluonField.leg(p3, labels={LORENTZ_KIND: mu3, COLOR_ADJ_KIND: a3}),
)
LEGS_mixed_photon = (
    PsiMixField.leg(p1, conjugated=True, spin=s1, labels={SPINOR_KIND: i1, COLOR_FUND_KIND: c1}),
    PsiMixField.leg(p2, spin=s2, labels={SPINOR_KIND: i2, COLOR_FUND_KIND: c2}),
    GaugeField.leg(p3, labels={LORENTZ_KIND: mu3}),
)

# species_map is only needed for delta simplification; here we keep it minimal.
V_mixed_0 = model_vertex(compiled_mixed[0], LEGS_mixed_gluon, simplify_gamma=True)
V_mixed_1 = model_vertex(compiled_mixed[1], LEGS_mixed_photon, simplify_gamma=True)
print("=====Dirac kinetic (QCD + QED): gluon piece=====")
show("Vertex q̄ q G (SU(3) term)  [expected: -i g_S γ^μ T^a]", V_mixed_0)
print("=====Dirac kinetic (QCD + QED): photon piece=====")
show("Vertex q̄ q A (U(1) term)  [expected: -i e Q γ^μ × δ_color(c_1,c_2)]", V_mixed_1)




Mixed-group Dirac kinetic term expands into:
 - i PsiMixbar gamma^mu D_mu PsiMix [ColorFund]
 - i PsiMixbar gamma^mu D_mu PsiMix
=====Dirac kinetic (QCD + QED): gluon piece=====
=== Vertex q̄ q G (SU(3) term)  [expected: -i g_S γ^μ T^a] ===
-1𝑖*gS*gamma(bis(4, i1),bis(4, i2),mink(4, mu3))*t(coad(8, a3),cof(3, c1),cof(3, c2))
=====Dirac kinetic (QCD + QED): photon piece=====
=== Vertex q̄ q A (U(1) term)  [expected: -i e Q γ^μ × δ_color(c_1,c_2)] ===
-1𝑖*eQED*qMix*g(cof(3, c1),cof(3, c2))*gamma(bis(4, i1),bis(4, i2),mink(4, mu3))


### Mixed-group complex scalar kinetic term

The scalar analogue of the mixed QCD+QED fermion example is now implemented too.

For one complex scalar carrying both an SU(3) representation and a U(1) charge, the covariant compiler generates:

- the non-abelian current pieces
- the abelian current pieces
- the ordered mixed contact pieces

After summing the ordered mixed contacts, the physical four-point vertex has the expected factor `2 i g_S e Q g^{mu nu} T^a`.


In [16]:
# --- Mixed QCD + QED complex scalar kinetic term: currents + mixed contact ---
phiMix0, phiMixdag0 = S("phiMix0", "phiMixdag0")
qPhiMix = S("qPhiMix")
PhiMixField = Field(
    "PhiMix",
    spin=0,
    self_conjugate=False,
    symbol=phiMix0,
    conjugate_symbol=phiMixdag0,
    indices=(COLOR_FUND_INDEX,),
    quantum_numbers={"Q": qPhiMix},
)
MODEL_MIXED_SCALAR_COV = Model(
    name="MixedScalarQCDQED-covariant",
    gauge_groups=(QCD_GROUP, QED_GROUP),
    fields=(PhiMixField, GluonField, GaugeField),
    covariant_terms=(ComplexScalarKineticTerm(field=PhiMixField),),
)
compiled_mixed_scalar = compile_covariant_terms(MODEL_MIXED_SCALAR_COV)
print("\nMixed-group complex-scalar kinetic term expands into:")
for lab in labels(compiled_mixed_scalar):
    print(" -", lab)

mixed_scalar_qcd_terms = [t for t in compiled_mixed_scalar if "SU3C: scalar current" in t.label]
mixed_scalar_qed_terms = [t for t in compiled_mixed_scalar if "U1QED: scalar current" in t.label]
mixed_scalar_contact_terms = [t for t in compiled_mixed_scalar if "mixed contact" in t.label]

LEGS_mixed_scalar_gluon = (
    PhiMixField.leg(p1, conjugated=True, species=b1, labels={COLOR_FUND_KIND: c1}),
    PhiMixField.leg(p2, species=b2, labels={COLOR_FUND_KIND: c2}),
    GluonField.leg(p3, labels={LORENTZ_KIND: mu3, COLOR_ADJ_KIND: a3}, species=b3),
)
LEGS_mixed_scalar_photon = (
    PhiMixField.leg(p1, conjugated=True, species=b1, labels={COLOR_FUND_KIND: c1}),
    PhiMixField.leg(p2, species=b2, labels={COLOR_FUND_KIND: c2}),
    GaugeField.leg(p3, labels={LORENTZ_KIND: mu3}, species=b3),
)
LEGS_mixed_scalar_contact = (
    PhiMixField.leg(p1, conjugated=True, species=b1, labels={COLOR_FUND_KIND: c1}),
    PhiMixField.leg(p2, species=b2, labels={COLOR_FUND_KIND: c2}),
    GluonField.leg(p3, labels={LORENTZ_KIND: mu3, COLOR_ADJ_KIND: a3}, species=b3),
    GaugeField.leg(p4, labels={LORENTZ_KIND: mu4}, species=b4),
)

mixed_scalar_species_gluon = {b1: phiMixdag0, b2: phiMix0, b3: G0}
mixed_scalar_species_photon = {b1: phiMixdag0, b2: phiMix0, b3: A0}
mixed_scalar_species_contact = {b1: phiMixdag0, b2: phiMix0, b3: G0, b4: A0}

V_mixed_scalar_gluon = sum(
    (model_vertex(t, LEGS_mixed_scalar_gluon, species_map=mixed_scalar_species_gluon) for t in mixed_scalar_qcd_terms),
    Expression.num(0),
).expand()
V_mixed_scalar_photon = sum(
    (model_vertex(t, LEGS_mixed_scalar_photon, species_map=mixed_scalar_species_photon) for t in mixed_scalar_qed_terms),
    Expression.num(0),
).expand()
V_mixed_scalar_contact = sum(
    (model_vertex(t, LEGS_mixed_scalar_contact, species_map=mixed_scalar_species_contact) for t in mixed_scalar_contact_terms),
    Expression.num(0),
).expand()

print("=====Complex-scalar kinetic term (QCD + QED) - gluon current=====")
show("Vertex phi^dagger phi G from mixed-group (D_mu phi)^dagger(D^mu phi)  [expected: i g_S (p_2-p_1)^mu T^a]", V_mixed_scalar_gluon)
print("=====Complex-scalar kinetic term (QCD + QED) - photon current=====")
show("Vertex phi^dagger phi A from mixed-group (D_mu phi)^dagger(D^mu phi)  [expected: i e Q (p_2-p_1)^mu x delta_color(c_1,c_2)]", V_mixed_scalar_photon)
print("=====Complex-scalar kinetic term (QCD + QED) - mixed contact=====")
show("Vertex phi^dagger phi G A from mixed-group (D_mu phi)^dagger(D^mu phi)  [expected: 2 i g_S e Q g^{mu nu} T^a]", V_mixed_scalar_contact)



Mixed-group complex-scalar kinetic term expands into:
 - (D_mu PhiMix)^dagger (D^mu PhiMix) SU3C: scalar current (+)
 - (D_mu PhiMix)^dagger (D^mu PhiMix) SU3C: scalar current (-)
 - (D_mu PhiMix)^dagger (D^mu PhiMix) SU3C: scalar contact [slots 1,1]
 - (D_mu PhiMix)^dagger (D^mu PhiMix) U1QED: scalar current (+)
 - (D_mu PhiMix)^dagger (D^mu PhiMix) U1QED: scalar current (-)
 - (D_mu PhiMix)^dagger (D^mu PhiMix) U1QED: scalar contact
 - (D_mu PhiMix)^dagger (D^mu PhiMix) SU3C x U1QED: scalar mixed contact [slot 1]
 - (D_mu PhiMix)^dagger (D^mu PhiMix) U1QED x SU3C: scalar mixed contact [slot 1]
=====Complex-scalar kinetic term (QCD + QED) - gluon current=====
=== Vertex phi^dagger phi G from mixed-group (D_mu phi)^dagger(D^mu phi)  [expected: i g_S (p_2-p_1)^mu T^a] ===
-1𝑖*gS*t(coad(8, a3),cof(3, c1),cof(3, c2))*pcomp(p1,mu)+1𝑖*gS*t(coad(8, a3),cof(3, c1),cof(3, c2))*pcomp(p2,mu)
=====Complex-scalar kinetic term (QCD + QED) - photon current=====
=== Vertex phi^dagger phi A from mixe

In [17]:
# --- Bislotted scalar with explicit opt-in: slot_policy='sum' ---
COLOR_FUND_REP_SUM = GaugeRepresentation(
    index=COLOR_FUND_INDEX,
    generator_builder=gauge_generator,
    name="fundamental_sum",
    slot_policy="sum",
)
QCD_GROUP_BISLOT_SUM = GaugeGroup(
    name="SU3CBiSum",
    abelian=False,
    coupling=gS,
    gauge_boson=G0,
    structure_constant=structure_constant,
    representations=(COLOR_FUND_REP_SUM,),
)
MODEL_BISLOT_SUM_COV = Model(
    name="ScalarQCD-bislot-covariant-sum",
    gauge_groups=(QCD_GROUP_BISLOT_SUM,),
    fields=(PhiBiField, GluonField),
    covariant_terms=(ComplexScalarKineticTerm(field=PhiBiField),),
)
compiled_bislot_sum = compile_covariant_terms(MODEL_BISLOT_SUM_COV)
print("\nBislotted scalar covariant expansion (slot_policy='sum') generated labels:")
for lab in labels(compiled_bislot_sum):
    print(" -", lab)

LEGS_bislot_current = (
    PhiBiField.leg(p1, conjugated=True, species=b1, labels={COLOR_FUND_KIND: (c1, c3)}),
    PhiBiField.leg(p2, species=b2, labels={COLOR_FUND_KIND: (c2, c4)}),
    GluonField.leg(p3, labels={LORENTZ_KIND: mu3, COLOR_ADJ_KIND: a3}, species=b3),
)
LEGS_bislot_contact = (
    PhiBiField.leg(p1, conjugated=True, species=b1, labels={COLOR_FUND_KIND: (c1, c3)}),
    PhiBiField.leg(p2, species=b2, labels={COLOR_FUND_KIND: (c2, c4)}),
    GluonField.leg(p3, labels={LORENTZ_KIND: mu3, COLOR_ADJ_KIND: a3}, species=b3),
    GluonField.leg(p4, labels={LORENTZ_KIND: mu4, COLOR_ADJ_KIND: a4}, species=b4),
)

bislot_species_3 = {b1: phiBidag0, b2: phiBi0, b3: G0}
bislot_species_4 = {b1: phiBidag0, b2: phiBi0, b3: G0, b4: G0}

V_bislot_current_sum = sum(
    (model_vertex(t, LEGS_bislot_current, species_map=bislot_species_3) for t in compiled_bislot_sum if "current" in t.label),
    Expression.num(0),
).expand()
V_bislot_contact_sum = sum(
    (model_vertex(t, LEGS_bislot_contact, species_map=bislot_species_4) for t in compiled_bislot_sum if "contact" in t.label),
    Expression.num(0),
).expand()

print("=====Bislot scalar (D_μ φ)†(D^μ φ), slot_policy='sum' — summed currents=====")
show("Summed vertex φ† φ G (both color slots)  [expected: i g_S × Σ_slots (p_2-p_1)^μ T^a with δ on spectator slot]", V_bislot_current_sum)
print("=====Bislot scalar — summed contacts (ordered slot pairs)=====")
show("Summed vertex φ† φ G G  [expected: i g_S^2 g^{μν} × (same-slot chains + cross-slot T⊗T)]", V_bislot_contact_sum)


Bislotted scalar covariant expansion (slot_policy='sum') generated labels:
 - (D_mu PhiBi)^dagger (D^mu PhiBi) SU3CBiSum: scalar current (+)_slot1
 - (D_mu PhiBi)^dagger (D^mu PhiBi) SU3CBiSum: scalar current (-)_slot1
 - (D_mu PhiBi)^dagger (D^mu PhiBi) SU3CBiSum: scalar current (+)_slot2
 - (D_mu PhiBi)^dagger (D^mu PhiBi) SU3CBiSum: scalar current (-)_slot2
 - (D_mu PhiBi)^dagger (D^mu PhiBi) SU3CBiSum: scalar contact [slots 1,1]
 - (D_mu PhiBi)^dagger (D^mu PhiBi) SU3CBiSum: scalar contact [slots 1,2]
 - (D_mu PhiBi)^dagger (D^mu PhiBi) SU3CBiSum: scalar contact [slots 2,1]
 - (D_mu PhiBi)^dagger (D^mu PhiBi) SU3CBiSum: scalar contact [slots 2,2]
=====Bislot scalar (D_μ φ)†(D^μ φ), slot_policy='sum' — summed currents=====
=== Summed vertex φ† φ G (both color slots)  [expected: i g_S × Σ_slots (p_2-p_1)^μ T^a with δ on spectator slot] ===
-1𝑖*gS*g(cof(3, c1),cof(3, c2))*t(coad(8, a3),cof(3, c3),cof(3, c4))*pcomp(p1,mu)+1𝑖*gS*g(cof(3, c1),cof(3, c2))*t(coad(8, a3),cof(3, c3),cof(3, 

## 10. Pure-gauge sector and Yang-Mills self-interactions

The latest implemented layer is the pure-gauge kinetic sector.

A single `GaugeKineticTerm(...)` can now generate:

- the gauge bilinear
- the Yang-Mills cubic vertex
- the Yang-Mills quartic vertex

for the covered non-abelian case.

In [18]:
MODEL_QCD_GAUGE_COV = Model(
    name="QCDGauge-covariant",
    gauge_groups=(QCD_GROUP,),
    fields=(GluonField,),
    gauge_kinetic_terms=(GaugeKineticTerm(gauge_group=QCD_GROUP),),
)

compiled_ym = compile_covariant_terms(MODEL_QCD_GAUGE_COV)

print("Number of generated pure-gauge interactions:", len(compiled_ym))
print("Generated labels:")
for label in labels(compiled_ym):
    print(" -", label)

LEGS_two_gluon = (
    GluonField.leg(p1, species=b1, labels={LORENTZ_KIND: mu, COLOR_ADJ_KIND: a3}),
    GluonField.leg(p2, species=b2, labels={LORENTZ_KIND: nu, COLOR_ADJ_KIND: a4}),
)
LEGS_three_gluon = (
    GluonField.leg(p1, species=b1, labels={LORENTZ_KIND: mu, COLOR_ADJ_KIND: a3}),
    GluonField.leg(p2, species=b2, labels={LORENTZ_KIND: nu, COLOR_ADJ_KIND: a4}),
    GluonField.leg(p3, species=b3, labels={LORENTZ_KIND: rho, COLOR_ADJ_KIND: a5}),
)
LEGS_four_gluon = (
    GluonField.leg(p1, species=b1, labels={LORENTZ_KIND: mu, COLOR_ADJ_KIND: a3}),
    GluonField.leg(p2, species=b2, labels={LORENTZ_KIND: nu, COLOR_ADJ_KIND: a4}),
    GluonField.leg(p3, species=b3, labels={LORENTZ_KIND: rho, COLOR_ADJ_KIND: a5}),
    GluonField.leg(p4, species=b4, labels={LORENTZ_KIND: sigma, COLOR_ADJ_KIND: a6}),
)

V_ym_0 = model_vertex(
    compiled_ym[0],
    LEGS_two_gluon,
    species_map={b1: G0, b2: G0},
)
V_ym_1 = model_vertex(
    compiled_ym[1],
    LEGS_two_gluon,
    species_map={b1: G0, b2: G0},
)
V_ym_cubic = model_vertex(
    compiled_ym[2],
    LEGS_three_gluon,
    species_map={b1: G0, b2: G0, b3: G0},
)
V_ym_4 = model_vertex(
    compiled_ym[3],
    LEGS_four_gluon,
    species_map={b1: G0, b2: G0, b3: G0, b4: G0},
)

V_ym_0 = simplify_gamma_chain(V_ym_0)
V_ym_1 = simplify_gamma_chain(V_ym_1)   
V_ym_cubic = simplify_gamma_chain(V_ym_cubic)
V_ym_4 = simplify_gamma_chain(V_ym_4)

show("Yang-Mills three-gluon vertex from GaugeKineticTerm (first term)", V_ym_0)
show("Yang-Mills three-gluon vertex from GaugeKineticTerm (second term)", V_ym_1)
show("Yang-Mills cubic vertex from GaugeKineticTerm", V_ym_cubic)
show("Yang-Mills four-gluon vertex from GaugeKineticTerm", V_ym_4)

Number of generated pure-gauge interactions: 4
Generated labels:
 - -1/4 SU3C field strength squared SU3C: gauge kinetic bilinear (metric)
 - -1/4 SU3C field strength squared SU3C: gauge kinetic bilinear (cross)
 - -1/4 SU3C field strength squared SU3C: Yang-Mills cubic
 - -1/4 SU3C field strength squared SU3C: Yang-Mills quartic
=== Yang-Mills three-gluon vertex from GaugeKineticTerm (first term) ===
1𝑖*g(mink(4, mu),mink(4, nu))*g(coad(8, a3),coad(8, a4))*pcomp(p1,rho_G_SU3C)*pcomp(p2,rho_G_SU3C)
=== Yang-Mills three-gluon vertex from GaugeKineticTerm (second term) ===
-1𝑖/2*g(coad(8, a3),coad(8, a4))*g(mink(4, mu),mink(4, rho_left_G_SU3C))*g(mink(4, nu),mink(4, rho_right_G_SU3C))*pcomp(p1,rho_right_G_SU3C)*pcomp(p2,rho_left_G_SU3C)-1𝑖/2*g(coad(8, a3),coad(8, a4))*g(mink(4, mu),mink(4, rho_right_G_SU3C))*g(mink(4, nu),mink(4, rho_left_G_SU3C))*pcomp(p1,rho_left_G_SU3C)*pcomp(p2,rho_right_G_SU3C)
=== Yang-Mills cubic vertex from GaugeKineticTerm ===
1𝑖*(-1𝑖*gS*f(coad(8, a3),coad(8, a4

## 10.5 Ordinary Gauge Fixing And Ghosts

This section adds the ordinary gauge-fixing and ghost pieces in the same simple style as sections 9.1 and 10.

We keep the same pattern:

- define a small model
- compile the terms
- print the generated labels and compiled `InteractionTerm` objects
- evaluate the corresponding vertex
- show the compact readability form for the same result


In [ ]:
from model import GaugeFixingTerm, GhostTerm
from operators import gauge_fixing_bilinear, gauge_kinetic_bilinear, ghost_gauge, ghost_kinetic

xiQED, xiQCD = S("xiQED", "xiQCD")
ghG0, ghGbar0 = S("ghG0", "ghGbar0")

GhostGluonField = Field(
    "ghG",
    spin=0,
    kind="ghost",
    self_conjugate=False,
    symbol=ghG0,
    conjugate_symbol=ghGbar0,
    indices=(COLOR_ADJ_INDEX,),
)

QCD_GROUP_GHOST = GaugeGroup(
    name="SU3C",
    abelian=False,
    coupling=gS,
    gauge_boson=G0,
    ghost_field=ghG0,
    structure_constant=structure_constant,
    representations=(COLOR_FUND_REP,),
)

MODEL_QED_GAUGE_FIXING = Model(
    name="QEDGaugeFixing-covariant",
    gauge_groups=(QED_GROUP,),
    fields=(GaugeField,),
    gauge_fixing_terms=(GaugeFixingTerm(gauge_group=QED_GROUP, xi=xiQED),),
)

compiled_qed_gf = compile_covariant_terms(MODEL_QED_GAUGE_FIXING)
print("QED gauge-fixing terms:", labels(compiled_qed_gf))
print("Compiled object:")
print(compiled_qed_gf[0])

LEGS_qed_gf = (
    GaugeField.leg(p1, species=b1, labels={LORENTZ_KIND: mu3}),
    GaugeField.leg(p2, species=b2, labels={LORENTZ_KIND: mu4}),
)

V_qed_gf = model_vertex(
    compiled_qed_gf[0],
    LEGS_qed_gf,
    species_map={b1: A0, b2: A0},
)

print("=====Gauge fixing (QED)=====")
show("Vertex from -(1/2 xi) (partial.A)^2", V_qed_gf)
show("Compact form", (I / xiQED) * gauge_fixing_bilinear(mu3, mu4, p1, p2))

MODEL_QCD_GAUGE_FIXING = Model(
    name="QCDGaugeFixing-covariant",
    gauge_groups=(QCD_GROUP,),
    fields=(GluonField,),
    gauge_fixing_terms=(GaugeFixingTerm(gauge_group=QCD_GROUP, xi=xiQCD),),
)

compiled_qcd_gf = compile_covariant_terms(MODEL_QCD_GAUGE_FIXING)
print("QCD gauge-fixing terms:", labels(compiled_qcd_gf))
print("Compiled object:")
print(compiled_qcd_gf[0])

LEGS_qcd_gf = (
    GluonField.leg(p1, species=b1, labels={LORENTZ_KIND: mu3, COLOR_ADJ_KIND: a3}),
    GluonField.leg(p2, species=b2, labels={LORENTZ_KIND: mu4, COLOR_ADJ_KIND: a4}),
)

V_qcd_gf = model_vertex(
    compiled_qcd_gf[0],
    LEGS_qcd_gf,
    species_map={b1: G0, b2: G0},
)

print("=====Gauge fixing (QCD)=====")
show("Vertex from -(1/2 xi) (partial.G)^2", V_qcd_gf)
show(
    "Compact form",
    (I / xiQCD) * gauge_fixing_bilinear(mu3, mu4, p1, p2) * COLOR_ADJ_INDEX.representation.g(a3, a4).to_expression(),
)

MODEL_QED_GAUGE_FIXED = Model(
    name="QEDGaugeFixed-covariant",
    gauge_groups=(QED_GROUP,),
    fields=(GaugeField,),
    gauge_kinetic_terms=(GaugeKineticTerm(gauge_group=QED_GROUP),),
    gauge_fixing_terms=(GaugeFixingTerm(gauge_group=QED_GROUP, xi=xiQED),),
)

compiled_qed_gauge_fixed = compile_covariant_terms(MODEL_QED_GAUGE_FIXED)
print("QED gauge-fixed terms:", labels(compiled_qed_gauge_fixed))

V_qed_gauge_fixed = simplify_gamma_chain(
    model_vertex(compiled_qed_gauge_fixed[0], LEGS_qed_gf, species_map={b1: A0, b2: A0})
    + model_vertex(compiled_qed_gauge_fixed[1], LEGS_qed_gf, species_map={b1: A0, b2: A0})
    + model_vertex(compiled_qed_gauge_fixed[2], LEGS_qed_gf, species_map={b1: A0, b2: A0})
).expand()

photon_rho = compiled_qed_gauge_fixed[0].derivatives[0].lorentz_index

print("=====Ordinary gauge-fixed photon bilinear=====")
show("Full two-point photon vertex", V_qed_gauge_fixed)
show(
    "Compact form",
    I * (
        gauge_kinetic_bilinear(mu3, mu4, p1, p2, photon_rho)
        + gauge_fixing_bilinear(mu3, mu4, p1, p2) / xiQED
    ),
)

MODEL_QCD_GHOST = Model(
    name="QCDGhost-covariant",
    gauge_groups=(QCD_GROUP_GHOST,),
    fields=(GluonField, GhostGluonField),
    ghost_terms=(GhostTerm(gauge_group=QCD_GROUP_GHOST),),
)

compiled_qcd_ghost = compile_covariant_terms(MODEL_QCD_GHOST)
print("QCD ghost terms:", labels(compiled_qcd_ghost))
print("Compiled ghost bilinear object:")
print(compiled_qcd_ghost[0])
print("Compiled ghost-gluon object:")
print(compiled_qcd_ghost[1])

LEGS_ghost_bilinear = (
    GhostGluonField.leg(p1, conjugated=True, species=b1, labels={COLOR_ADJ_KIND: a3}),
    GhostGluonField.leg(p2, species=b2, labels={COLOR_ADJ_KIND: a4}),
)

LEGS_ghost_gluon = (
    GhostGluonField.leg(p1, conjugated=True, species=b1, labels={COLOR_ADJ_KIND: a3}),
    GluonField.leg(p2, species=b2, labels={LORENTZ_KIND: mu3, COLOR_ADJ_KIND: a4}),
    GhostGluonField.leg(p3, species=b3, labels={COLOR_ADJ_KIND: a5}),
)

V_ghost_bilinear = model_vertex(
    compiled_qcd_ghost[0],
    LEGS_ghost_bilinear,
    species_map={b1: ghGbar0, b2: ghG0},
)

V_ghost_gluon = model_vertex(
    compiled_qcd_ghost[1],
    LEGS_ghost_gluon,
    species_map={b1: ghGbar0, b2: G0, b3: ghG0},
)

print("=====Ghost bilinear=====")
show("Vertex from the ghost bilinear", V_ghost_bilinear)
show("Compact form", -I * ghost_kinetic(a3, a4, p1, p2, S("rho_ghost")))

print("=====Ghost-gluon interaction=====")
show("Vertex from the ghost-gluon term", V_ghost_gluon)
show("Compact form", -gS * ghost_gauge(a3, a4, a5, mu3, p1))

MODEL_QCD_GAUGE_FIXED = Model(
    name="QCDGaugeFixed-covariant",
    gauge_groups=(QCD_GROUP_GHOST,),
    fields=(GluonField, GhostGluonField),
    gauge_kinetic_terms=(GaugeKineticTerm(gauge_group=QCD_GROUP_GHOST),),
    gauge_fixing_terms=(GaugeFixingTerm(gauge_group=QCD_GROUP_GHOST, xi=xiQCD),),
    ghost_terms=(GhostTerm(gauge_group=QCD_GROUP_GHOST),),
)

compiled_qcd_gauge_fixed = compile_covariant_terms(MODEL_QCD_GAUGE_FIXED)
print("QCD gauge-fixed terms:", labels(compiled_qcd_gauge_fixed))

V_qcd_gauge_fixed = simplify_gamma_chain(
    model_vertex(compiled_qcd_gauge_fixed[0], LEGS_qcd_gf, species_map={b1: G0, b2: G0})
    + model_vertex(compiled_qcd_gauge_fixed[1], LEGS_qcd_gf, species_map={b1: G0, b2: G0})
    + model_vertex(compiled_qcd_gauge_fixed[4], LEGS_qcd_gf, species_map={b1: G0, b2: G0})
).expand()

gluon_rho = compiled_qcd_gauge_fixed[0].derivatives[0].lorentz_index

print("=====Ordinary gauge-fixed gluon bilinear=====")
show("Full two-point gluon vertex", V_qcd_gauge_fixed)
show(
    "Compact form",
    I
    * (
        gauge_kinetic_bilinear(mu3, mu4, p1, p2, gluon_rho)
        + gauge_fixing_bilinear(mu3, mu4, p1, p2) / xiQCD
    )
    * COLOR_ADJ_INDEX.representation.g(a3, a4).to_expression(),
)


## 11. FeynRules-style Lagrangian API

Everything above uses the low-level pipeline: manually build `InteractionTerm` + `ExternalLeg` objects, call `vertex_factor(...)`, simplify.

The **Lagrangian API** wraps all of that into a single entry point that mirrors FeynRules:

1. Compose interaction terms with `+` (including `InteractionTerm + InteractionTerm`).
2. Call `Model.lagrangian()` to compile all declared sectors into a `Lagrangian` object.
3. Call `L.feynman_rule(field1, field2, ...)` — legs, momenta (`q1, q2, ...`), and indices (`i1, i2, ...`) are assigned automatically.

Use `field.bar` for conjugated fields (anti-particles).

Below we replicate every sector already demonstrated in this notebook, but now through the Lagrangian API.

In [ ]:
from model import Lagrangian, GaugeFixingTerm, GhostTerm, ConjugateField

# --- 11a. Scalar phi^4 via Lagrangian ---

PhiField_L = Field("Phi", spin=0, self_conjugate=True, symbol=S("phi"))
lam4 = S("lam4")

term_phi4 = InteractionTerm(
    coupling=lam4,
    fields=tuple(PhiField_L.occurrence() for _ in range(4)),
)
L_phi4_lagr = Lagrangian(terms=(term_phi4,))
V_phi4_lagr = L_phi4_lagr.feynman_rule(PhiField_L, PhiField_L, PhiField_L, PhiField_L)
show("phi^4 via Lagrangian API", V_phi4_lagr)

# --- 11b. Complex scalar mass term ---

PhiC = Field("PhiC", spin=0, self_conjugate=False, symbol=S("phiC"), conjugate_symbol=S("phiCdag"))
lamC = S("lamC")

term_phiC = InteractionTerm(
    coupling=lamC,
    fields=(PhiC.occurrence(conjugated=True), PhiC.occurrence()),
)
L_phiC = Lagrangian(terms=(term_phiC,))
V_phiC = L_phiC.feynman_rule(PhiC.bar, PhiC)
show("phiC^dag phiC mass term via Lagrangian API", V_phiC)

# --- 11c. InteractionTerm + InteractionTerm composition ---

term_a = InteractionTerm(coupling=S("g1"), fields=tuple(PhiField_L.occurrence() for _ in range(4)))
term_b = InteractionTerm(coupling=S("g2"), fields=tuple(PhiField_L.occurrence() for _ in range(4)))
L_sum = term_a + term_b
print(f"\nInteractionTerm + InteractionTerm -> {type(L_sum).__name__} with {len(L_sum.terms)} terms")
V_sum = L_sum.feynman_rule(PhiField_L, PhiField_L, PhiField_L, PhiField_L)
show("Summed phi^4 vertex", V_sum)

In [ ]:
# --- 11d. QCD fermion-gluon vertex via Model.lagrangian() ---

MODEL_QCD_FERMION_L = Model(
    gauge_groups=(QCD_GROUP,),
    fields=(QuarkField, GluonField),
    covariant_terms=(DiracKineticTerm(field=QuarkField),),
)

L_qcd_fermion = MODEL_QCD_FERMION_L.lagrangian()
print(f"QCD fermion Lagrangian: {len(L_qcd_fermion.terms)} terms")
V_qcd_fermion = L_qcd_fermion.feynman_rule(QuarkField.bar, QuarkField, GluonField)
show("QCD quark-gluon vertex via Lagrangian API", V_qcd_fermion)

# Cross-check: matches the low-level pipeline result
compiled_qcd_ref = compile_covariant_terms(MODEL_QCD_FERMION_L)
legs_qcd_ref = (
    QuarkField.leg(S("q1"), conjugated=True, labels={SPINOR_KIND: S("i1"), COLOR_FUND_KIND: S("i2")}),
    QuarkField.leg(S("q2"), labels={SPINOR_KIND: S("i3"), COLOR_FUND_KIND: S("i4")}),
    GluonField.leg(S("q3"), labels={LORENTZ_KIND: S("i5"), COLOR_ADJ_KIND: S("i6")}),
)
V_qcd_ref = simplify_vertex(vertex_factor(
    interaction=compiled_qcd_ref[0], external_legs=legs_qcd_ref,
    x=S("x_"), d=S("d"), strip_externals=True, include_delta=True,
))
assert V_qcd_fermion.expand().to_canonical_string() == V_qcd_ref.expand().to_canonical_string()
print("\n  Cross-check vs low-level pipeline: PASS")

In [ ]:
# --- 11e. Gauge kinetic, gauge-fixing, and ghost sectors via Lagrangian ---

MODEL_QCD_FULL = Model(
    name="QCDFull-Lagrangian",
    gauge_groups=(QCD_GROUP_GHOST,),
    fields=(GluonField, GhostGluonField),
    gauge_kinetic_terms=(GaugeKineticTerm(gauge_group=QCD_GROUP_GHOST),),
    gauge_fixing_terms=(GaugeFixingTerm(gauge_group=QCD_GROUP_GHOST, xi=xiQCD),),
    ghost_terms=(GhostTerm(gauge_group=QCD_GROUP_GHOST),),
)

L_qcd_full = MODEL_QCD_FULL.lagrangian()
print(f"QCD full Lagrangian: {len(L_qcd_full.terms)} terms")

# Yang-Mills bilinear (2-gluon)
V_gluon_bilinear = L_qcd_full.feynman_rule(GluonField, GluonField)
show("Gauge-fixed gluon bilinear (kinetic + gauge-fixing)", V_gluon_bilinear)

# 3-gluon vertex
V_3g = L_qcd_full.feynman_rule(GluonField, GluonField, GluonField)
show("3-gluon vertex", V_3g)

# 4-gluon vertex
V_4g = L_qcd_full.feynman_rule(GluonField, GluonField, GluonField, GluonField)
show("4-gluon vertex", V_4g)

# Ghost bilinear
V_ghost_bi = L_qcd_full.feynman_rule(GhostGluonField.bar, GhostGluonField)
show("Ghost bilinear", V_ghost_bi)

# Ghost-gluon 3-point
V_ghost_gluon = L_qcd_full.feynman_rule(GhostGluonField.bar, GluonField, GhostGluonField)
show("Ghost-gluon vertex", V_ghost_gluon)

# --- 11f. Precompiled-model idempotency ---
from gauge_compiler import with_compiled_covariant_terms
L_precompiled = with_compiled_covariant_terms(MODEL_QCD_FULL).lagrangian()
assert len(L_precompiled.terms) == len(L_qcd_full.terms), "Double-count bug!"
V_3g_pre = L_precompiled.feynman_rule(GluonField, GluonField, GluonField)
assert V_3g.expand().to_canonical_string() == V_3g_pre.expand().to_canonical_string()
print("\n  Precompiled idempotency cross-check: PASS")

## 12. What this code can currently do

By the end of this notebook we have seen the main implemented workflow:

1. **Direct engine usage** for scalar and fermion interaction terms
2. **Derivative handling** through momentum factors
3. **Open spinor and Lorentz index handling** through explicit labels
4. **Structured model-layer declarations** using `Field`, `ExternalLeg`, `InteractionTerm`, and `Model`
5. **Minimal gauge compilation** from metadata
6. **Repeated same-kind slot handling** when the active slot is declared explicitly with `GaugeRepresentation(slot=...)`
7. **Convention-fixed covariant compilation** for matter kinetic terms, including mixed-group scalar contacts
8. **Pure-gauge field-strength compilation** up to Yang-Mills 3- and 4-point vertices
9. **Ordinary linear-covariant gauge fixing** for abelian and non-abelian gauge fields
10. **Ordinary non-abelian Faddeev-Popov ghosts** in the covered unbroken case
11. **FeynRules-style Lagrangian API**: `Model.lagrangian()` + `L.feynman_rule(...)` with automatic leg, momentum, and index conventions

### Current codebase strengths

- one central engine drives both direct and model-based workflows
- the tensor vocabulary is explicit and reusable
- the gauge compiler is already able to derive standard matter, pure-gauge, gauge-fixing, and ghost interactions from metadata
- the Lagrangian API provides a single-command interface matching the FeynRules workflow
- the code is organized by layer rather than by isolated examples

### Current limitations

- multi-fermion tensor support is still narrower than a full FeynRules-like system
- repeated identical index kinds now work better, but ambiguous same-kind slots still require explicit `GaugeRepresentation(slot=...)`
- background-field-specific splitting and the broader BFM layer are not yet implemented
- the covariant derivative is not yet a first-class object that automatically expands based on field symmetries

### Recommended next reading in the source tree

- `src/model.py`: the data model (including `Lagrangian`, `ConjugateField`)
- `src/model_symbolica.py`: the engine
- `src/gauge_compiler.py`: the compiler path from model metadata to explicit interaction terms
- `src/examples.py`: the full demo and regression matrix
- `src/spenso_structures.py`: the tensor building blocks
- `src/tensor_canonicalization.py`: tensor-head and dummy-index canonicalization helpers

This notebook is meant to make those files much easier to read.


## 13. What This Notebook Does Not Cover From `src/examples.py`

If you read only this notebook, you should understand the architecture and the main workflow, but not every runnable check in `src/examples.py`.

Still mainly only in `src/examples.py`:

- direct/model cross-checks across the full covered set
- larger regression coverage for fermion operators and role filtering
- tensor canonicalization checks
- the full covariant, pure-gauge, gauge-fixing, and ghost assertion matrix
- the complete validation harness that the repository currently uses

So the notebook is the best entry point, while `src/examples.py` remains the executable reference.
